In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.enableHiveSupport() \
.master("yarn") \
.getOrCreate()

In [3]:
from pyspark.sql import functions as F

bank_login_df = spark.range(50000) \
        .withColumn("user_id", (F.rand() * 500).cast("int")) \
        .withColumn("login_timestamp", F.to_timestamp(F.lit("2025-11-01")) + F.expr("rand() * 10000 * interval 1 minute")) \
        .withColumn("ip_address", F.expr("concat('192.168.1.', cast(rand() * 255 as int))"))

In [4]:
bank_login_df.show(truncate=False)

+---+-------+--------------------------+-------------+
|id |user_id|login_timestamp           |ip_address   |
+---+-------+--------------------------+-------------+
|0  |406    |2025-11-01 04:52:50.233826|192.168.1.102|
|1  |408    |2025-11-04 21:54:24.400309|192.168.1.143|
|2  |220    |2025-11-06 11:53:06.659293|192.168.1.38 |
|3  |105    |2025-11-02 08:54:23.425694|192.168.1.250|
|4  |258    |2025-11-04 07:08:14.480464|192.168.1.164|
|5  |216    |2025-11-01 18:08:54.958953|192.168.1.17 |
|6  |488    |2025-11-05 07:50:22.731632|192.168.1.168|
|7  |242    |2025-11-05 01:36:29.958632|192.168.1.82 |
|8  |409    |2025-11-06 19:38:42.247226|192.168.1.7  |
|9  |163    |2025-11-06 15:15:55.636476|192.168.1.106|
|10 |125    |2025-11-01 12:01:16.905509|192.168.1.92 |
|11 |16     |2025-11-06 20:35:31.392639|192.168.1.108|
|12 |104    |2025-11-02 10:40:08.947877|192.168.1.8  |
|13 |480    |2025-11-05 16:39:20.144169|192.168.1.64 |
|14 |259    |2025-11-01 06:29:52.774869|192.168.1.183|
|15 |497  

In [5]:
from pyspark.sql import Window
window_spec = Window.partitionBy("user_id").orderBy(F.asc("login_timestamp"))

In [6]:
login_info_df = bank_login_df.withColumn("last_login", F.lag("login_timestamp", 1).over(window_spec))

In [7]:
login_info_df.show(truncate=False)

+-----+-------+--------------------------+-------------+--------------------------+
|id   |user_id|login_timestamp           |ip_address   |last_login                |
+-----+-------+--------------------------+-------------+--------------------------+
|26557|148    |2025-11-01 02:15:36.860996|192.168.1.137|null                      |
|14067|148    |2025-11-01 03:03:23.063008|192.168.1.242|2025-11-01 02:15:36.860996|
|683  |148    |2025-11-01 03:03:30.34219 |192.168.1.123|2025-11-01 03:03:23.063008|
|27784|148    |2025-11-01 03:29:47.68313 |192.168.1.182|2025-11-01 03:03:30.34219 |
|41223|148    |2025-11-01 06:08:08.412883|192.168.1.66 |2025-11-01 03:29:47.68313 |
|24987|148    |2025-11-01 07:49:48.315298|192.168.1.171|2025-11-01 06:08:08.412883|
|17348|148    |2025-11-01 09:03:32.152952|192.168.1.18 |2025-11-01 07:49:48.315298|
|44638|148    |2025-11-01 10:02:27.640611|192.168.1.18 |2025-11-01 09:03:32.152952|
|1460 |148    |2025-11-01 12:57:04.703606|192.168.1.52 |2025-11-01 10:02:27.

In [10]:
login_diff = login_info_df.withColumn("login_gap", F.col("login_timestamp")-F.col("last_login"))

In [11]:
login_diff.show(truncate=False)

+-----+-------+--------------------------+-------------+--------------------------+------------------------------------+
|id   |user_id|login_timestamp           |ip_address   |last_login                |login_gap                           |
+-----+-------+--------------------------+-------------+--------------------------+------------------------------------+
|26557|148    |2025-11-01 02:15:36.860996|192.168.1.137|null                      |null                                |
|14067|148    |2025-11-01 03:03:23.063008|192.168.1.242|2025-11-01 02:15:36.860996|47 minutes 46.202012 seconds        |
|683  |148    |2025-11-01 03:03:30.34219 |192.168.1.123|2025-11-01 03:03:23.063008|7.279182 seconds                    |
|27784|148    |2025-11-01 03:29:47.68313 |192.168.1.182|2025-11-01 03:03:30.34219 |26 minutes 17.34094 seconds         |
|41223|148    |2025-11-01 06:08:08.412883|192.168.1.66 |2025-11-01 03:29:47.68313 |2 hours 38 minutes 20.729753 seconds|
|24987|148    |2025-11-01 07:49: